In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, row_number, when
from pyspark.sql.window import Window
import numpy as np
import pandas as pd
import json

spark = SparkSession.builder \
    .appName("Pertemuan7-StudiKasusTerintegrasi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

26/09/28 10:57:23 WARN Utils: Your hostname, tarin resolves to a loopback address: 127.0.1.1; using 192.168.1.18 instead (on interface wlp0s20f3)
26/09/28 10:57:23 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/28 10:57:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/28 10:57:25 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


SparkSession siap. Versi Spark: 3.5.9


In [3]:
import numpy as np
import pandas as pd
import json

np.random.seed(123)

# Sumber 1: Data cabang (JSON)
cabang_list = ["Kopi Nusantara Magelang", "Kopi Nusantara Yogyakarta", "Kopi Nusantara Semarang"]
data_cabang = [
    {"cabang_id": i + 1, "nama_cabang": c, "kota": c.split()[-1],
     "kepala_barista": np.random.choice(["Wulan", "Reza", "Nadia"])}
    for i, c in enumerate(cabang_list)
]
with open("tugas7_cabang.json", "w") as f:
    for c in data_cabang:
        f.write(json.dumps(c) + "\n")

# Sumber 2: Data menu (CSV)
menu_list = [
    {"menu_id": i, "nama_menu": f"Menu-{i}",
     "kategori_menu": np.random.choice(["Kopi", "Non-Kopi", "Makanan Ringan"]),
     "harga": int(np.random.choice([18000, 22000, 25000, 30000, 35000]))}
    for i in range(1, 16)
]
pd.DataFrame(menu_list).to_csv("tugas7_menu.csv", index=False)

# Sumber 3: Data transaksi (CSV) — data utama
n = 4000
data_transaksi_tugas7 = pd.DataFrame({
    "trx_id": [f"KN{i}" for i in range(n)],
    "cabang_id": np.random.randint(1, 4, size=n),
    "menu_id": np.random.randint(1, 16, size=n),
    "qty": np.random.randint(1, 5, size=n),
})
data_transaksi_tugas7.to_csv("tugas7_transaksi.csv", index=False)

!hdfs dfs -mkdir -p /tarin/home/Praktikum-BigData/tugas7/raw
!hdfs dfs -put -f tugas7_transaksi.csv /tarin/home/Praktikum-BigData/tugas7/raw/
!hdfs dfs -put -f tugas7_cabang.json /tarin/home/Praktikum-BigData/tugas7/raw/
!hdfs dfs -put -f tugas7_menu.csv /tarin/home/Praktikum-BigData/tugas7/raw/

print("Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.")
print(f"- Cabang: {len(data_cabang)} | Menu: {len(menu_list)} | Transaksi: {n}")

Ketiga sumber data siap & data transaksi sudah diunggah ke HDFS.
- Cabang: 3 | Menu: 15 | Transaksi: 4000


In [4]:
spark = SparkSession.builder.appName("Tugas7-KopiNusantara").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

base = "hdfs://localhost:9000/tarin/home/Praktikum-BigData/tugas7/raw/"
df_transaksi = spark.read.csv(base + "tugas7_transaksi.csv", header=True, inferSchema=True)
df_cabang    = spark.read.json(base + "tugas7_cabang.json")
df_menu      = spark.read.csv(base + "tugas7_menu.csv", header=True, inferSchema=True)

print(df_transaksi.count(), df_cabang.count(), df_menu.count())   # 4000, 3, 15

4000 3 15


In [5]:
df_gabungan = df_transaksi.join(df_cabang, on="cabang_id", how="inner") \
                          .join(df_menu, on="menu_id", how="inner") \
                          .withColumn("total_penjualan", col("qty") * col("harga"))
df_gabungan.show(5)

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


In [6]:
ringkas = df_gabungan.groupBy("nama_cabang", "nama_menu") \
    .agg(spark_sum("total_penjualan").alias("total_penjualan"))

w = Window.partitionBy("nama_cabang").orderBy(col("total_penjualan").desc())

top2 = ringkas.withColumn("peringkat", row_number().over(w)) \
              .filter(col("peringkat") <= 2)
top2.orderBy("nama_cabang", "peringkat").show()

+--------------------+---------+---------------+---------+
|         nama_cabang|nama_menu|total_penjualan|peringkat|
+--------------------+---------+---------------+---------+
|Kopi Nusantara Ma...|   Menu-1|        8505000|        1|
|Kopi Nusantara Ma...|  Menu-12|        8085000|        2|
|Kopi Nusantara Se...|  Menu-13|        8750000|        1|
|Kopi Nusantara Se...|   Menu-9|        8715000|        2|
|Kopi Nusantara Yo...|  Menu-12|        9275000|        1|
|Kopi Nusantara Yo...|  Menu-15|        8160000|        2|
+--------------------+---------+---------------+---------+



In [7]:
df_gabungan.createOrReplaceTempView("gabungan")

spark.sql("""
    SELECT kepala_barista, SUM(total_penjualan) AS total_penjualan
    FROM gabungan
    GROUP BY kepala_barista
    ORDER BY total_penjualan DESC
""").show()

+--------------+---------------+
|kepala_barista|total_penjualan|
+--------------+---------------+
|         Nadia|      177680000|
|          Reza|       84645000|
+--------------+---------------+



In [8]:
out = "hdfs://localhost:9000/tarin/home/Praktikum-BigData/tugas7/processed/laporan"
df_gabungan.write.mode("overwrite").partitionBy("kategori_menu").parquet(out)

df_cek = spark.read.parquet(out)
print("Jumlah baris:", df_cek.count())   # harus 4000
!hdfs dfs -ls /tarin/home/Praktikum-BigData/tugas7/processed/laporan

Jumlah baris: 4000
Found 4 items
-rw-r--r--   3 tarin supergroup          0 2026-09-28 10:59 /tarin/home/Praktikum-BigData/tugas7/processed/laporan/_SUCCESS
drwxr-xr-x   - tarin supergroup          0 2026-09-28 10:59 /tarin/home/Praktikum-BigData/tugas7/processed/laporan/kategori_menu=Kopi
drwxr-xr-x   - tarin supergroup          0 2026-09-28 10:59 /tarin/home/Praktikum-BigData/tugas7/processed/laporan/kategori_menu=Makanan Ringan
drwxr-xr-x   - tarin supergroup          0 2026-09-28 10:59 /tarin/home/Praktikum-BigData/tugas7/processed/laporan/kategori_menu=Non-Kopi


Buat tabel ringkasan (per cabang: total penjualan, jumlah transaksi, rata-rata nilai transaksi), lalu tulis pada markdown cell (minimal 120 kata): cabang mana yang berkinerja terbaik, menu apa yang sebaiknya dipromosikan lebih gencar di cabang dengan kinerja terlemah (berdasarkan temuan bagian C), dan alasannya.

In [9]:
ringkasan = df_cek.groupBy("nama_cabang", "kota", "kepala_barista").agg(
    spark_sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_nilai_transaksi")
).orderBy(col("total_penjualan").desc())
ringkasan.show(truncate=False)

+-------------------------+----------+--------------+---------------+----------------+-------------------------+
|nama_cabang              |kota      |kepala_barista|total_penjualan|jumlah_transaksi|rata_rata_nilai_transaksi|
+-------------------------+----------+--------------+---------------+----------------+-------------------------+
|Kopi Nusantara Semarang  |Semarang  |Nadia         |90542000       |1393            |64997.8463747308         |
|Kopi Nusantara Magelang  |Magelang  |Nadia         |87138000       |1302            |66926.26728110599        |
|Kopi Nusantara Yogyakarta|Yogyakarta|Reza          |84645000       |1305            |64862.06896551724        |
+-------------------------+----------+--------------+---------------+----------------+-------------------------+



Berdasarkan data dari tabel diatas, Cabang Kopi Nusantara Semarang memiliki kinerja terbaik dengan total penjualan sebesar 90 juta dan jumlah transaksi sebanyak 1393. Kemudian terdapat cabang di kota Magelang dengan total penjualan sebesar 87 juta dan jumlah transaksi sebanyak 1302. Dan cabang yang paling rendah dari total penjualan yaitu cabang Yogyakarta. Namun dari data tabel tersebut, jika dilihat dari rata-rata nilai transaksi, ketiga cabang relatif sama. Lalu jika dilihat dari bagian C, menu dengan total penjualan tertinggi di cabang Semarang ialah menu 13 dan menu 9. Sedangkan di cabang Magelang, menu dengan penjualan tertinggi ialah menu 1 dan menu 12. Dan jika dilihat dari cabang paling lemah yaitu cabang Yogyakarta, ialah menu 12 dan menu 15. Yang berarti dapat ditarik kesimpulan, karena menu 12 dan menu 15 adalah dua menu dengan total penjualan tertinggi di Yogyakarta, keduanya perlu dipertahankan dan dipromosikan lebih gencar untuk menjaga pelanggan lama. Untuk menarik pelanggan baru, Yogyakarta dapat mencoba mempromosikan menu 1 yang terbukti laris di Magelang, karena menu tersebut belum masuk dua besar di Yogyakarta.